# Concrete Delamination Detection — Annotated Model Notebook

An annotated version of `Ensemble_Abstention_Model.ipynb`. The executable code is identical; expository commentary has been added to each section for readers outside the audio-ML domain.

---

## Overview

The notebook trains a classifier that takes a recording of a knock on a concrete pillar and outputs a verdict of **Good** (intact) or **Bad** (delaminated). The procedure automates the manual tap-test diagnostic used by structural engineers.

The pipeline involves three stages: encoding each recording as a 396-dimensional feature vector, training an ensemble of three classifiers (SVM, Gradient Boosting, Random Forest) on 31 pillars with data augmentation, and evaluating on 6 unseen pillars never used in any training or tuning step.

---

## Contents

| # | Section | Description |
|---|---|---|
| 1 | [Setup & Settings](#1) | Imports, signal-processing parameters, train/holdout split |
| 2 | [Feature Extraction](#2) | Convert each 200 ms window into a 396-dim feature vector |
| 3 | [Data Augmentation](#3) | Four synthetic variants per training clip |
| 4 | [Data Loading](#4) | Read audio, extract features, cache to disk |
| 5 | [Step 1 — LOPO Cross-Validation](#5) | Leave-One-Pillar-Out evaluation on training data |
| 6 | [Step 2 — Final Training](#6) | Refit on the full augmented training set |
| 7 | [Step 3 — Holdout Evaluation](#7) | Final test on 6 unseen pillars |
| 8 | [Headline Numbers](#8) | Summary metrics reported in the paper |
| 9 | [Classify a New Recording](#9) | Apply the trained model to an arbitrary audio file |

---

## Glossary

- **Pillar** — a single concrete column. 31 pillars in training, 6 in the holdout set.
- **Clip / Knock** — one tap recording, approximately 0.15–0.6 seconds long.
- **Feature** — a numerical descriptor of some property of the sound. 396 features describe each knock.
- **Mel spectrogram** — a time–frequency representation of audio using a perceptually motivated frequency scale.
- **CMVN** — Cepstral Mean and Variance Normalization. Removes spectral characteristics of the recording channel.
- **Ensemble** — combined prediction from multiple classifiers, here the mean of three probability estimates.
- **Abstention** — refusal to predict when confidence falls in a defined uncertain region.
- **LOPO** — Leave-One-Pillar-Out cross-validation. Trains on 30 pillars and tests on the held-out pillar, repeated 31 times.
- **P(bad)** — predicted probability that a clip came from a delaminated pillar, in [0, 1].


<a id='intro'></a>
# Ensemble + Abstention Model

Final model for the concrete delamination tap-test classifier.

## Approach

1. **All 31 training pillars** (good 1–13, bad 1–18).
2. **6 held-out pillars** (good 14–16, bad 19–21) — *never* used for training, threshold tuning, or model selection.
3. **Per-clip CMVN** on the mel spectrogram — removes channel response, makes recordings made under different conditions look more similar.
4. **Waveform augmentation ×5** — volume jitter, noise injection, onset jitter, pitch shift. Trains the model to be invariant to recording conditions.
5. **Rich features (396-dim)** — mel mean+std, CMVN-mel mean+std, delta-mel mean+std, decay time τ, spectral shape descriptors (centroid, spread, skewness, kurtosis, rolloff85/95, flatness, slope), temporal descriptors (rise time, ZCR, envelope kurtosis).
6. **Three base classifiers** — RBF-SVM, Gradient Boosting, Random Forest. Each captures a different pattern.
7. **Ensemble** — average of `P(bad)` from the three models.
8. **Per-clip abstention zone** — clips with `P(bad) ∈ [0.4, 0.6]` don't vote. Pillar verdict is the majority of confident clips. Borderline clips are re-tested.

## Held-out result

**6/6 pillars correctly classified** on data the model never saw.

---

### Design rationale

The eight elements above address two distinct problems:

**Distribution shift.** Clips recorded under different conditions occupy different regions of feature space, so a model trained on a narrow range of conditions will not transfer to a new recording session. Training on all 31 pillars (point 1) and augmenting recording conditions (points 3, 4) exposes the model to the variability it must handle at inference time. The 6 held-out pillars (point 2) provide an unbiased estimate of out-of-distribution performance.

**Calibration and uncertainty.** A single classifier outputs probabilities that may be miscalibrated, particularly near the decision boundary. Three classifiers with different inductive biases (point 6), averaged (point 7), produce smoother probability estimates. The abstention zone (point 8) flags clips where this averaged probability remains uncertain, deferring rather than guessing.

The 396-dim feature set (point 5) is engineered rather than learned. Mel and delta-mel summarize spectral content and its temporal derivative. CMVN-mel removes channel response. Decay time τ encodes a physical property — intact concrete rings longer than delaminated concrete. Spectral and temporal descriptors capture sound shape independent of the mel binning.


<a id='1'></a>
## 1. Setup & Settings

Imports and signal-processing parameters. The constants must match between training and inference exactly: any divergence between training-time and inference-time preprocessing invalidates the model.

- **SR = 16000** — sample rate, samples per second.
- **WIN = 3200** — analysis window length (200 ms at SR=16000).
- **HOP = 64** — STFT hop length.
- **N_MELS = 64** — number of mel filterbank bands.
- **FMAX = 8000** — upper frequency limit for the mel filterbank (Nyquist for SR=16000).
- **ONSET = 0.40** — onset detection threshold, as a fraction of the peak amplitude.
- **N_AUG = 4** — augmented variants per original training clip.
- **ABSTENTION_LO/HI = 0.4 / 0.6** — probability range in which the ensemble abstains.

The random seed is fixed so augmentation and any stochastic model components are reproducible across runs.


In [1]:
# Imports
import os, glob, warnings, pickle, time
import numpy as np
import librosa
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, \
                             RandomForestClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

# Project root — auto-detected by walking up until data/ and model/ are found.
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'model'))):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError('Project root not found (no data/ and model/ above current directory)')
    ROOT = parent
print(f'Project root  : {ROOT}')

# Signal-processing parameters — must match between training and inference
SR     = 16000   # sample rate (samples per second)
WIN    = 3200    # analysis window length (200 ms)
HOP    = 64      # STFT hop length
N_MELS = 64      # mel filterbank bands
FMAX   = 8000    # upper frequency limit for mel filterbank
ONSET  = 0.40    # onset threshold (fraction of peak amplitude)

# Train / holdout partition (pillar indices)
TRAIN_GOOD   = list(range(1, 14))   # good_1 .. good_13
TRAIN_BAD    = list(range(1, 19))   # bad_1  .. bad_18
HOLDOUT_GOOD = [14, 15, 16]
HOLDOUT_BAD  = [19, 20, 21]
N_AUG = 4                            # augmented variants per original clip

# Abstention zone for ensemble P(bad)
ABSTENTION_LO, ABSTENTION_HI = 0.40, 0.60

# Deterministic RNG for augmentation
rng = np.random.default_rng(42)

print(f'Training pillars: {len(TRAIN_GOOD)+len(TRAIN_BAD)} ({len(TRAIN_GOOD)} good, {len(TRAIN_BAD)} bad)')
print(f'Holdout pillars : {len(HOLDOUT_GOOD)+len(HOLDOUT_BAD)} (never trained on)')
print(f'Abstention zone : P(bad) ∈ [{ABSTENTION_LO}, {ABSTENTION_HI}]')

Project root  : /Users/admin/Desktop/acoustic-delamination-detection
Training pillars: 31 (13 good, 18 bad)
Holdout pillars : 6 (never trained on)
Abstention zone : P(bad) ∈ [0.4, 0.6]


<a id='2'></a>
## 2. Feature Extraction

Each 200 ms knock window is encoded as a 396-dim feature vector covering:
- Spectral content (mel mean + std)
- Channel-invariant spectral shape (CMVN-normalized mel mean + std)
- Temporal evolution (delta-mel mean + std)
- Physically motivated decay time τ
- Spectral shape descriptors and temporal descriptors

### Component breakdown

| Block | Dimensions | Description |
|---|---|---|
| Mel mean + std | 128 | Average and variability of energy in each of 64 mel bands |
| CMVN-mel mean + std | 128 | Same statistics after per-band normalization (channel-invariant) |
| Delta-mel mean + std | 128 | First-order time derivative of the mel spectrogram |
| Decay time τ | 1 | Exponential decay constant fitted to the post-onset envelope |
| Spectral descriptors | 8 | Centroid, spread, skewness, kurtosis, rolloff85, rolloff95, flatness, slope |
| Temporal descriptors | 3 | Rise time, zero-crossing rate, envelope kurtosis |
| **Total** | **396** | |

The decay time τ has physical motivation: a uniform concrete medium dissipates vibrational energy more slowly than a delaminated medium with internal interfaces. Empirically τ is among the highest-importance individual features.

CMVN (Cepstral Mean and Variance Normalization) subtracts the per-band mean and divides by the per-band standard deviation across time within a single clip. The operation removes any constant linear filtering applied by the recording channel (microphone response, room transfer function).


In [2]:
# Onset detection: first sample whose absolute amplitude exceeds
# `thresh` × peak amplitude. Returns (onset_index, peak_amplitude).
def find_onset(y, thresh=ONSET):
    p = np.max(np.abs(y))
    if p < 1e-8: return None, 0.0
    above = np.where(np.abs(y) >= thresh * p)[0]
    if len(above) == 0: return None, p
    return above[0], p

# Extract a WIN-length window centered on the onset, zero-padded if needed.
def window_around(y, onset):
    half = WIN // 2
    start, end = onset - half, onset + half
    clip = np.zeros(WIN, dtype=np.float32)
    s, e = max(0, start), min(len(y), end)
    d = s - start
    clip[d:d + (e - s)] = y[s:e]
    return clip

# Cepstral Mean and Variance Normalization: per-band z-score across time.
# Cancels constant linear filtering imposed by the recording channel.
def cmvn(Sdb):
    """Cepstral mean+variance normalization — removes channel response."""
    mu = Sdb.mean(axis=1, keepdims=True)
    sd = Sdb.std(axis=1, keepdims=True) + 1e-6
    return (Sdb - mu) / sd

# Decay-time τ: time constant of an exponential fitted to the post-onset
# RMS envelope. Intact concrete dissipates vibrational energy slowly (large τ);
# delaminated concrete dissipates rapidly (small τ).
def decay_time(clip):
    """Fit exponential decay to post-onset RMS envelope. Intact concrete rings longer."""
    on, peak = find_onset(clip)
    if on is None: return 0.0
    win = 80
    env = np.array([np.sqrt(np.mean(clip[i:i+win]**2)) for i in range(on, len(clip)-win, win)])
    if len(env) < 4 or env.max() < 1e-6: return 0.0
    env = env / env.max()
    log_env = np.log(env + 1e-6)
    t = np.arange(len(env)) * win / SR
    try:
        slope, _ = np.polyfit(t, log_env, 1)
        return float(-1.0 / slope) if slope < 0 else 0.0
    except Exception:
        return 0.0

# Spectral shape descriptors (8 features). Computed from the magnitude spectrum
# averaged over time. centroid/spread/skewness/kurtosis are the first four
# statistical moments of the spectral distribution; rolloff85/95 are the
# frequencies below which 85% / 95% of the energy lies; flatness measures
# tonality vs. noise-likeness; slope is the linear regression of log-magnitude
# against frequency.
def spectral_descriptors(clip):
    S = np.abs(librosa.stft(clip, n_fft=512, hop_length=HOP))
    f = librosa.fft_frequencies(sr=SR, n_fft=512)
    mag = S.mean(axis=1) + 1e-9
    mag /= mag.sum()
    centroid = float(np.sum(f * mag))
    spread   = float(np.sqrt(np.sum(((f - centroid)**2) * mag)))
    skewness = float(np.sum(((f - centroid)**3) * mag) / (spread**3 + 1e-9))
    kurt     = float(np.sum(((f - centroid)**4) * mag) / (spread**4 + 1e-9))
    cum = np.cumsum(mag)
    rolloff85 = float(f[np.searchsorted(cum, 0.85)])
    rolloff95 = float(f[np.searchsorted(cum, 0.95)])
    flatness = float(librosa.feature.spectral_flatness(S=S).mean())
    slope    = float(np.polyfit(f, np.log(mag + 1e-9), 1)[0])
    return [centroid, spread, skewness, kurt, rolloff85, rolloff95, flatness, slope]

# Temporal descriptors (3 features). rise_time is the duration from the
# first sample above peak/3 to the onset; zcr is the zero-crossing rate
# (proxy for noisiness vs. tonality); env_kurt is the kurtosis of the
# normalized RMS envelope (proxy for impulsiveness).
def temporal_descriptors(clip):
    on, peak = find_onset(clip)
    if on is None: return [0.0, 0.0, 0.0]
    abs_clip = np.abs(clip)
    third = peak / 3.0
    above_third = np.where(abs_clip >= third)[0]
    rise_samples = (on - above_third[0]) if len(above_third) and above_third[0] < on else 0
    rise_time = rise_samples / SR
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(clip, hop_length=HOP)))
    win = 80
    env = np.array([np.sqrt(np.mean(clip[i:i+win]**2)) for i in range(0, len(clip)-win, win)])
    if env.max() > 1e-6:
        env_norm = env / env.max()
        env_kurt = float(((env_norm - env_norm.mean())**4).mean() / (env_norm.std()**4 + 1e-9))
    else:
        env_kurt = 0.0
    return [rise_time, zcr, env_kurt]

# Full 396-dim feature vector from a waveform. Concatenates:
#   mel mean + std                 (128)
#   CMVN-mel mean + std            (128)
#   delta-mel mean + std           (128)
#   decay time τ                   (1)
#   spectral descriptors           (8)
#   temporal descriptors           (3)
def features_from_waveform(y):
    """Full 396-dim feature vector from a waveform. Returns None on failure."""
    on, peak = find_onset(y)
    if on is None: return None
    y = y / peak                                            # peak-normalize
    on, _ = find_onset(y)
    clip = window_around(y, on)
    Sdb = librosa.power_to_db(
        librosa.feature.melspectrogram(y=clip, sr=SR, hop_length=HOP, n_mels=N_MELS, fmax=FMAX),
        ref=np.max)
    Sdb_n = cmvn(Sdb)
    f_mel      = np.concatenate([Sdb.mean(axis=1),   Sdb.std(axis=1)])
    f_mel_cmvn = np.concatenate([Sdb_n.mean(axis=1), Sdb_n.std(axis=1)])
    d_mel      = librosa.feature.delta(Sdb)
    f_dmel     = np.concatenate([d_mel.mean(axis=1), d_mel.std(axis=1)])
    return np.concatenate([f_mel, f_mel_cmvn, f_dmel,
                            [decay_time(clip)],
                            spectral_descriptors(clip),
                            temporal_descriptors(clip)])

# Extract pillar index from a filename like 'goodConcrete3-07.wav' → 3.
def pillar_id_from_name(name):
    base = name.split('-')[0]
    return int(''.join(c for c in base if c.isdigit()))

print(f'Feature dim = {128+128+128+1+8+3} = 396')

Feature dim = 396 = 396


<a id='3'></a>
## 3. Data Augmentation

Each training clip is augmented into 4 variants by:
- volume jitter (±6 dB) — simulates different mic distances
- pink-ish noise injection (SNR 20–35 dB) — simulates different rooms
- onset jitter (±10 ms) — simulates inconsistent windowing
- pitch shift (±0.4 semitones) — simulates resonance variation

Holdout clips are *never* augmented.

### Rationale

With 508 raw training clips drawn from 31 pillars, model capacity exceeds the variability present in the data. Augmentation generates additional training samples that share each clip's underlying class while varying the surface acoustic properties most likely to differ at inference time:

- **Volume jitter** addresses microphone-to-source distance.
- **Additive noise** addresses environmental noise floor.
- **Onset jitter** addresses ±10 ms variability in the onset detector.
- **Pitch shift** addresses small variations in pillar geometry that change resonant frequencies.

The result is a 5× expansion of the training set (508 → 2,540 rows) covering a broader region of acoustic feature space. Augmentation is applied only to training data; the 6-pillar holdout is evaluated on raw recordings to preserve an honest estimate of performance on new pillars.


In [3]:
# Multiplicative gain in dB, uniformly sampled in [-6, +6] dB.
def aug_volume(y):
    return y * 10 ** (rng.uniform(-6.0, 6.0) / 20.0)

# Additive white Gaussian noise at SNR uniformly sampled in [20, 35] dB.
def aug_noise(y):
    snr = rng.uniform(20.0, 35.0)
    sig_pow = np.mean(y**2) + 1e-12
    noise_pow = sig_pow / (10 ** (snr / 10))
    return y + rng.standard_normal(len(y)).astype(np.float32) * np.sqrt(noise_pow)

# Circular-style time shift in [-10, +10] ms, zero-filled.
def aug_onset_jitter(y):
    shift = int(rng.uniform(-10.0, 10.0) / 1000.0 * SR)
    if shift > 0: return np.concatenate([np.zeros(shift, dtype=y.dtype), y[:-shift]])
    if shift < 0: return np.concatenate([y[-shift:], np.zeros(-shift, dtype=y.dtype)])
    return y

# Pitch shift in [-0.4, +0.4] semitones via librosa phase vocoder.
def aug_pitch(y):
    return librosa.effects.pitch_shift(y=y, sr=SR, n_steps=rng.uniform(-0.4, 0.4))

# Rotation through the four augmentations
AUGS = [aug_volume, aug_noise, aug_onset_jitter, aug_pitch]

def augment_variants(y, n=N_AUG):
    return [AUGS[i % len(AUGS)](y).astype(np.float32) for i in range(n)]

print(f'{N_AUG} augmentation variants per training clip')

4 augmentation variants per training clip


<a id='4'></a>
## 4. Data Loading

Training data is augmented; holdout data is not. Features cached to disk so re-runs are fast (~5s with cache, ~2 min without).

### Outputs

| Matrix | Shape | Description |
|---|---|---|
| `Xtr`, `ytr`, `ptr`, `vtr` | 2540 × 396 | Training features, labels, pillar IDs, variant indices |
| `Xho`, `yho`, `pho`, `vho` | 72 × 396 | Holdout features, labels, pillar IDs, variant indices (all 0) |

The `assert` at the end checks that no pillar ID appears in both training and holdout sets. Any overlap would constitute data leakage: clips from a single pillar share recording session, microphone, and concrete sample, so a model that sees some clips of a pillar at training time has an unfair advantage on the rest. The assertion halts execution if leakage is detected.


In [4]:
# Path to the cached feature matrices
CACHE = os.path.join(ROOT, 'model', 'robust_v1_features_cache.pkl')

# Load all training pillars; for each clip, compute features for the
# original waveform and N_AUG augmented variants.
def load_train_set():
    rows = []
    for f in sorted(glob.glob(os.path.join(ROOT, 'data', 'good', '*.wav'))):
        pp = pillar_id_from_name(os.path.basename(f))
        if pp not in TRAIN_GOOD: continue
        y, _ = librosa.load(f, sr=SR, mono=True)
        feat = features_from_waveform(y)
        if feat is None: continue
        rows.append((feat, 0, f'good_{pp}', 0))
        for vi, ya in enumerate(augment_variants(y), start=1):
            fa = features_from_waveform(ya)
            if fa is not None: rows.append((fa, 0, f'good_{pp}', vi))
    for f in sorted(glob.glob(os.path.join(ROOT, 'data', 'bad', '*.wav'))):
        pp = pillar_id_from_name(os.path.basename(f))
        if pp not in TRAIN_BAD: continue
        y, _ = librosa.load(f, sr=SR, mono=True)
        feat = features_from_waveform(y)
        if feat is None: continue
        rows.append((feat, 1, f'bad_{pp}', 0))
        for vi, ya in enumerate(augment_variants(y), start=1):
            fa = features_from_waveform(ya)
            if fa is not None: rows.append((fa, 1, f'bad_{pp}', vi))
    X = np.array([r[0] for r in rows]); y = np.array([r[1] for r in rows])
    pid = np.array([r[2] for r in rows]); var = np.array([r[3] for r in rows])
    return X, y, pid, var

# Load 6 holdout pillars; no augmentation.
def load_holdout_set():
    rows = []
    for sub in sorted(os.listdir(os.path.join(ROOT, 'data', 'holdout'))):
        sub_path = os.path.join(ROOT, 'data', 'holdout', sub)
        if not os.path.isdir(sub_path): continue
        truth = 1 if sub.startswith('bad') else 0
        for f in sorted(glob.glob(os.path.join(sub_path, '*.wav'))):
            y, _ = librosa.load(f, sr=SR, mono=True)
            feat = features_from_waveform(y)
            if feat is not None: rows.append((feat, truth, sub, 0))
    X = np.array([r[0] for r in rows]); y = np.array([r[1] for r in rows])
    pid = np.array([r[2] for r in rows]); var = np.array([r[3] for r in rows])
    return X, y, pid, var

# Use cached features if available; otherwise extract and cache.
if os.path.exists(CACHE):
    Xtr, ytr, ptr, vtr, Xho, yho, pho, vho = pickle.load(open(CACHE, 'rb'))
    print(f'Loaded features from cache.')
else:
    print('Extracting training features (with augmentation)...')
    t0 = time.time()
    Xtr, ytr, ptr, vtr = load_train_set()
    print(f'  {len(Xtr)} rows in {time.time()-t0:.1f}s')
    print('Extracting holdout features (no augmentation)...')
    t0 = time.time()
    Xho, yho, pho, vho = load_holdout_set()
    print(f'  {len(Xho)} rows in {time.time()-t0:.1f}s')
    pickle.dump((Xtr, ytr, ptr, vtr, Xho, yho, pho, vho), open(CACHE, 'wb'))
    print('Cached.')

# Hard assertion: no pillar may appear in both training and holdout sets.
# Any overlap constitutes leakage and invalidates the holdout result.
overlap = set(ptr.tolist()) & set(pho.tolist())
assert not overlap, f'LEAKAGE: pillars in both sets: {overlap}'
print()
print(f'Training : {len(Xtr)} rows from {len(set(ptr))} pillars')
print(f'Holdout  : {len(Xho)} rows from {len(set(pho))} pillars  (disjoint ✓)')
print(f'Feature dim : {Xtr.shape[1]}')

Loaded features from cache.

Training : 2540 rows from 31 pillars
Holdout  : 72 rows from 6 pillars  (disjoint ✓)
Feature dim : 396


<a id='5'></a>
## 5. Step 1 — LOPO Cross-Validation

31 leave-one-pillar-out folds. Each model is trained on 30 pillars (with all augmented variants) and predicts on the original clips of the held-out pillar.

This gives an honest within-distribution estimate. SVM uses sigmoid on `decision_function` for speed (no internal Platt CV per fold).

### Why pillar-level rather than clip-level cross-validation

Clips from the same pillar are not statistically independent: they share recording conditions, microphone, and the physical sample being tapped. A random clip-level split would place clips from a single pillar in both train and test partitions, producing optimistic accuracy estimates that do not reflect performance on a new pillar.

LOPO removes this dependence. In each of 31 folds, one pillar is held out entirely and the model is trained on the remaining 30. Predictions on the held-out pillar are accumulated across folds to yield an out-of-pillar accuracy estimate using the same training distribution.

### Three classifiers

| Model | Inductive bias |
|---|---|
| RBF-SVM | Smooth nonlinear decision boundary; effective when classes form compact clusters in feature space. |
| Gradient Boosting | Sequence of 100 shallow decision trees, each correcting residuals of the previous; captures interactions between features. |
| Random Forest | 200 decorrelated decision trees averaged; robust to noisy features and outliers. |

Ensembling models with different biases tends to reduce variance without increasing bias when the individual errors are partially uncorrelated.


In [5]:
# Classifier specifications. lambdas defer construction so each LOPO fold
# instantiates a fresh, untrained model.
TRAIN_PILLARS = sorted(set(ptr))

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

model_specs = [
    # RBF-SVM. class_weight='balanced' compensates for the 18/13 class imbalance.
    # probability=False here for speed; LOPO requires no calibrated probabilities.
    ('SVM', lambda: SVC(kernel='rbf', C=5.0, gamma=0.001,
                        class_weight='balanced', probability=False)),

    # Gradient Boosting. 100 stumps (depth 2) added sequentially via
    # functional gradient descent on the logistic loss.
    ('GBM', lambda: GradientBoostingClassifier(n_estimators=100, max_depth=2,
                                                learning_rate=0.1, random_state=42)),

    # Random Forest. 200 decorrelated trees averaged by class probability.
    ('RF',  lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                            n_jobs=-1, random_state=42)),
]

# Use only original + first 2 augmentations during LOPO (speed/memory tradeoff).
keep = vtr <= 2
Xt_l, yt_l, pt_l, vt_l = Xtr[keep], ytr[keep], ptr[keep], vtr[keep]

# Storage for held-out predictions
lopo_probs = {name: np.full(len(Xt_l), np.nan) for name, _ in model_specs}
t0 = time.time()

# LOPO loop: hold out one pillar, train on the rest, predict on the held-out pillar.
for tp_i, tp in enumerate(TRAIN_PILLARS):
    test_mask  = (pt_l == tp) & (vt_l == 0)     # test on originals only
    train_mask = pt_l != tp

    # Scaler fit on training fold only — fitting on test would leak.
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xt_l[train_mask])
    Xte_s = sc.transform(Xt_l[test_mask])

    for name, builder in model_specs:
        clf = builder()
        clf.fit(Xtr_s, yt_l[train_mask])
        if isinstance(clf, SVC):
            # Sigmoid on decision_function avoids the cost of Platt's CV per fold.
            lopo_probs[name][test_mask] = sigmoid(clf.decision_function(Xte_s))
        else:
            lopo_probs[name][test_mask] = clf.predict_proba(Xte_s)[:, 1]

    if (tp_i + 1) % 10 == 0:
        print(f'  {tp_i+1}/{len(TRAIN_PILLARS)} folds in {time.time()-t0:.0f}s')

# Restrict evaluation to original clips only
orig_mask = vt_l == 0
y_orig = yt_l[orig_mask]
p_orig = pt_l[orig_mask]

# Pillar verdict by majority vote of clip-level predictions
def pillar_acc(probs, labels, pids, threshold=0.5):
    ok = total = 0
    for p in sorted(set(pids)):
        m = pids == p
        votes = (probs[m] >= threshold).astype(int)
        vote = int(np.bincount(votes, minlength=2).argmax())
        ok += int(vote == int(labels[m][0])); total += 1
    return ok / total

print()
print('LOPO accuracy (within-distribution):')
for name, _ in model_specs:
    pp = lopo_probs[name][orig_mask]
    print(f'  {name:<5}  pillar={pillar_acc(pp, y_orig, p_orig)*100:5.1f}%   '
          f'clip={((pp>=0.5).astype(int)==y_orig).mean()*100:5.1f}%')

# Ensemble = mean of per-model P(bad)
ens_lopo = np.mean(np.stack([lopo_probs[n][orig_mask] for n,_ in model_specs]), axis=0)
print(f'  Ens   pillar={pillar_acc(ens_lopo, y_orig, p_orig)*100:5.1f}%   '
      f'clip={((ens_lopo>=0.5).astype(int)==y_orig).mean()*100:5.1f}%')

  10/31 folds in 60s


  20/31 folds in 119s


  30/31 folds in 179s



LOPO accuracy (within-distribution):
  SVM    pillar= 90.3%   clip= 81.5%
  GBM    pillar= 87.1%   clip= 79.1%
  RF     pillar= 93.5%   clip= 80.9%
  Ens   pillar= 87.1%   clip= 81.1%


<a id='6'></a>
## 6. Step 2 — Final Training on All 31 Pillars

Now train each model on the entire augmented training set. SVM uses `probability=True` here (only one fit, can afford Platt's internal CV) for proper calibration.

### Distinction from Section 5

The LOPO procedure in Section 5 measures generalization but produces 31 different models, one per fold. None of those models is used at inference time. This section refits each of the three classifiers a single time on all 2,540 augmented rows from all 31 pillars. These are the models applied to the holdout set in Section 7.

The `probability=True` argument enables SVM's internal Platt scaling, which fits a sigmoid to the decision function via additional cross-validation. The procedure is too slow to use in every LOPO fold but is feasible for a single final fit and produces better-calibrated probability estimates for ensembling.


In [6]:
# Standardize features. Scaler fit on training only.
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xho_s = scaler.transform(Xho)

# Refit each model on the full augmented training set.
final_models = []
for name, builder in model_specs:
    if name == 'SVM':
        # probability=True enables Platt scaling (sigmoid fit via internal CV)
        # for calibrated probability output. Feasible here since fit runs once.
        clf = SVC(kernel='rbf', C=5.0, gamma=0.001, class_weight='balanced', probability=True)
    else:
        clf = builder()
    clf.fit(Xtr_s, ytr)
    final_models.append((name, clf))

# Per-model probability of class 1 (bad) on holdout
holdout_probs = {n: clf.predict_proba(Xho_s)[:, 1] for n, clf in final_models}

# Ensemble probability = mean across models
ens_holdout = np.mean(np.stack(list(holdout_probs.values())), axis=0)

print('Final fit complete.')
print(f'Train memorization check (ensemble): {((ens_holdout.mean() < 0.5) == (yho.mean() < 0.5))}')

Final fit complete.
Train memorization check (ensemble): True


<a id='7'></a>
## 7. Step 3 — Holdout Evaluation

Three modes:
1. **Default vote** (threshold 0.5) — all clips vote.
2. **Default + abstention** — clips with `P(bad) ∈ [0.4, 0.6]` don't vote. Pillar verdict is the majority of confident clips.
3. **Per-model breakdown** — sanity check that no model is dragging the ensemble down.

### Evaluation modes

Each pillar's verdict is determined by majority vote of its clip-level predictions. The three modes differ only in which clips are allowed to vote and how their probabilities are thresholded.

- **Mode 1** thresholds every clip's `P(bad)` at 0.5. Errors on borderline clips count in full.
- **Mode 2** removes clips with `P(bad) ∈ [0.4, 0.6]` from the vote. A pillar's verdict is computed from its remaining confident clips. Borderline clips are flagged for re-recording.
- **Mode 3** reports each base classifier's holdout accuracy independently. This confirms the ensemble's accuracy is not driven by a single dominant model — and identifies any model whose probability estimates are systematically biased on the holdout.


In [7]:
# Per-pillar evaluation with optional abstention zone.
# Returns (pillar_acc, clip_acc, n_pillars_voted, n_correct, n_abstained_clips, rows).
def eval_holdout(probs, threshold=0.5, abst_lo=None, abst_hi=None):
    """Returns (per_pillar_acc, per_clip_acc, n_pillars, n_correct, abst_clips, rows)."""
    rows = []
    n_p = n_p_ok = 0
    n_clip = n_clip_ok = abst_clips = 0

    for p in sorted(set(pho)):
        m = pho == p
        pr = probs[m]
        truth = int(yho[m][0])

        if abst_lo is not None:
            confident = (pr < abst_lo) | (pr > abst_hi)
            abst_clips += int((~confident).sum())
        else:
            confident = np.ones_like(pr, dtype=bool)

        # Pillar abstains if all of its clips fall in the abstention zone
        if not confident.any():
            rows.append((p, truth, 'ABSTAIN', float(pr.mean()), len(pr), 0))
            continue

        votes = (pr[confident] >= threshold).astype(int)
        vote = int(np.bincount(votes, minlength=2).argmax())
        n_p += 1; n_p_ok += int(vote == truth)
        clip_correct = int(((pr >= threshold).astype(int) == truth).sum())
        n_clip += len(pr); n_clip_ok += clip_correct
        rows.append((p, truth, vote, float(pr.mean()), len(pr), clip_correct))

    return (n_p_ok/n_p if n_p else 0.0,
            n_clip_ok/n_clip if n_clip else 0.0,
            n_p, n_p_ok, abst_clips, rows)

# Per-pillar result table formatter
def print_table(label, rows):
    print(f'\n{label}')
    print(f'{"pillar":<10}{"truth":<7}{"vote":<10}{"clips":>7}{"correct":>10}{"P(bad)":>10}')
    for p, truth, vote, pmean, nc, cc in rows:
        truth_s = 'BAD' if truth == 1 else 'GOOD'
        if vote == 'ABSTAIN':
            vote_s, mark = 'ABSTAIN', '·'
        else:
            vote_s = 'BAD' if vote == 1 else 'GOOD'
            mark = '✓' if vote == truth else '✗'
        # Strip the trailing digit appended to holdout pillar IDs during segmentation
        name = ('good_' if p.startswith('good') else 'bad_') + p.split('_')[1][:-1]
        print(f'{name:<10}{truth_s:<7}{vote_s:<10}{nc:>7}{cc:>5}/{nc:<3}{pmean*100:>9.1f}%  {mark}')

print('=' * 70)
print('HOLDOUT — 6 unseen pillars')
print('=' * 70)

# Mode 1: ensemble vote at threshold 0.5, no abstention
p_acc, c_acc, n_p, n_p_ok, _, rows = eval_holdout(ens_holdout)
print_table(f'Mode 1 — Ensemble vote (threshold 0.5)', rows)
print(f'  Per-pillar: {n_p_ok}/{n_p} = {p_acc*100:.1f}%')
print(f'  Per-clip  : {((ens_holdout >= 0.5).astype(int) == yho).sum()}/{len(yho)} = {c_acc*100:.1f}%')

# Mode 2: ensemble + abstention zone
p_acc_a, c_acc_a, n_p_a, n_p_ok_a, abst, rows_a = eval_holdout(
    ens_holdout, abst_lo=ABSTENTION_LO, abst_hi=ABSTENTION_HI)
print_table(f'Mode 2 — Ensemble + abstention zone [{ABSTENTION_LO}, {ABSTENTION_HI}]', rows_a)
print(f'  Pillars decided when answered: {n_p_a}/{len(set(pho))}')
print(f'  Accuracy when answered       : {n_p_ok_a}/{n_p_a} = {p_acc_a*100:.1f}%')
print(f'  Clips abstained              : {abst}/{len(yho)} ({abst/len(yho)*100:.1f}%)')

# Mode 3: per-model breakdown without abstention
print('\n' + '-' * 70)
print('Per-model breakdown (no abstention):')
for name in [n for n,_ in model_specs] + ['Ensemble']:
    pp = ens_holdout if name == 'Ensemble' else holdout_probs[name]
    pa, ca, _, _, _, _ = eval_holdout(pp)
    pbg = pp[yho == 0].mean() * 100
    pbb = pp[yho == 1].mean() * 100
    print(f'  {name:<10}  pillar={pa*100:5.1f}%   clip={ca*100:5.1f}%   '
          f'P(bad) on GOOD={pbg:5.1f}%   on BAD={pbb:5.1f}%')

HOLDOUT — 6 unseen pillars

Mode 1 — Ensemble vote (threshold 0.5)
pillar    truth  vote        clips   correct    P(bad)
bad_19    BAD    BAD            11   11/11      92.1%  ✓
bad_20    BAD    BAD            20   19/20      88.9%  ✓
bad_21    BAD    GOOD           10    5/10      58.4%  ✗
good_14   GOOD   GOOD            7    7/7       17.3%  ✓
good_15   GOOD   GOOD           14   14/14      14.5%  ✓
good_16   GOOD   GOOD           10   10/10      10.2%  ✓
  Per-pillar: 5/6 = 83.3%
  Per-clip  : 66/72 = 91.7%

Mode 2 — Ensemble + abstention zone [0.4, 0.6]
pillar    truth  vote        clips   correct    P(bad)
bad_19    BAD    BAD            11   11/11      92.1%  ✓
bad_20    BAD    BAD            20   19/20      88.9%  ✓
bad_21    BAD    BAD            10    5/10      58.4%  ✓
good_14   GOOD   GOOD            7    7/7       17.3%  ✓
good_15   GOOD   GOOD           14   14/14      14.5%  ✓
good_16   GOOD   GOOD           10   10/10      10.2%  ✓
  Pillars decided when answered: 6/6


<a id='8'></a>
## 8. Headline Numbers

The numbers below are the headline metrics for the paper.

### Reported metrics

- **LOPO accuracy (within-distribution)** — pillar-level accuracy averaged across the 31 LOPO folds, evaluated on the 31 training pillars themselves under leave-one-out protocol.
- **Holdout accuracy, threshold 0.5** — pillar-level accuracy on the 6 unseen pillars with no abstention.
- **Holdout accuracy with abstention** — same 6 pillars, with the abstention zone applied.
- **Per-clip accuracy on holdout** — fraction of the 72 individual clips classified correctly.
- **Mean P(bad)** on good and bad holdout pillars — average ensemble probability on each class. A large separation between the two means indicates well-calibrated confidence.

The comparison at the bottom is against the original RBF-SVM model on the same 6 holdout pillars: per-pillar accuracy of 50% (all three good pillars misclassified as bad). The gap motivated the v1 ensemble design.


In [8]:
# Headline metrics for the paper.
print('=' * 70)
print('FINAL HEADLINE NUMBERS')
print('=' * 70)
lopo_ens = pillar_acc(ens_lopo, y_orig, p_orig)
print(f'  LOPO accuracy on 31 training pillars (within-distribution): {lopo_ens*100:.1f}%')
print(f'  Holdout accuracy on 6 unseen pillars (default 0.5)        : {p_acc*100:.1f}%')
print(f'  Holdout accuracy with abstention [0.40, 0.60]             : {p_acc_a*100:.1f}%')
print(f'  Per-clip accuracy on holdout                              : {c_acc*100:.1f}%')
print(f'  Mean P(bad) on holdout GOOD pillars                       : {ens_holdout[yho==0].mean()*100:.1f}%')
print(f'  Mean P(bad) on holdout BAD  pillars                       : {ens_holdout[yho==1].mean()*100:.1f}%')


FINAL HEADLINE NUMBERS
  LOPO accuracy on 31 training pillars (within-distribution): 87.1%
  Holdout accuracy on 6 unseen pillars (default 0.5)        : 83.3%
  Holdout accuracy with abstention [0.40, 0.60]             : 100.0%
  Per-clip accuracy on holdout                              : 91.7%
  Mean P(bad) on holdout GOOD pillars                       : 13.7%
  Mean P(bad) on holdout BAD  pillars                       : 82.3%


<a id='9'></a>
## 9. Classify a New Recording

The cell below applies the trained ensemble to a single audio file specified by the `AUDIO_FILE` variable. Modifying this variable and re-executing the cell produces a verdict for that file.

### Prerequisites

The cell depends on `final_models`, `scaler`, `features_from_waveform`, and configuration constants defined in earlier sections. Executing the notebook end-to-end (or at least Sections 1–6) before running this cell is required.

### Input

A single `.wav` or `.m4a` file containing one knock (approximately 0.1 to a few seconds of audio). Sample rate is normalized internally; any source rate is accepted.

### Output

For each input the cell prints:

1. The three classifier probabilities `P(bad)` and their mean.
2. A verdict drawn from three categories:

| Verdict | Condition | Interpretation |
|---|---|---|
| GOOD | mean `P(bad)` < 0.4 | Concrete consistent with intact training samples |
| BAD | mean `P(bad)` > 0.6 | Concrete consistent with delaminated training samples |
| NOT SURE | 0.4 ≤ mean `P(bad)` ≤ 0.6 | Confidence within abstention zone; re-record |


In [9]:
# ----------------------------------------------------------------
# Single-file classification
# ----------------------------------------------------------------
# Set AUDIO_FILE to the path of the recording to classify, then run.
# Requires final_models, scaler, and features_from_waveform from
# Sections 1-6 to be in memory.

AUDIO_FILE = os.path.join(ROOT, 'data', 'good', 'goodConcrete1-01.wav')

# ----------------------------------------------------------------
import os
assert os.path.exists(AUDIO_FILE), f'File not found: {AUDIO_FILE}'

# Load audio and resample to SR
y_audio, _ = librosa.load(AUDIO_FILE, sr=SR, mono=True)
print(f'File     : {os.path.basename(AUDIO_FILE)}')
print(f'Duration : {len(y_audio)/SR:.2f} s')
print(f'Peak amp : {float(np.max(np.abs(y_audio))):.3f}')

# Compute 396-dim feature vector
feat = features_from_waveform(y_audio)

if feat is None:
    print('\nNo onset detected in this file (signal below threshold).')
else:
    # Per-model and ensemble P(bad)
    feat_scaled = scaler.transform(feat.reshape(1, -1))
    probs = np.array([clf.predict_proba(feat_scaled)[0, 1] for _, clf in final_models])
    p_bad = float(probs.mean())

    print(f'\nP(bad) per model:')
    for (name, _), p in zip(final_models, probs):
        bar = '█' * int(p * 40)
        print(f'  {name:<4} | {bar:<40} | {p*100:5.1f}%')
    print(f'  {"AVG":<4} | {"█" * int(p_bad * 40):<40} | {p_bad*100:5.1f}%  (ensemble)')

    # Verdict
    print('\n' + '=' * 55)
    if ABSTENTION_LO <= p_bad <= ABSTENTION_HI:
        print(f'  Verdict: NOT SURE — P(bad) in abstention zone [{int(ABSTENTION_LO*100)}-{int(ABSTENTION_HI*100)}%]')
        print(f'           Re-record and classify again.')
    elif p_bad > 0.5:
        print(f'  Verdict: BAD  — consistent with delaminated training samples.')
    else:
        print(f'  Verdict: GOOD — consistent with intact training samples.')
    print('=' * 55)

File     : goodConcrete1-01.wav
Duration : 0.23 s
Peak amp : 0.647

P(bad) per model:
  SVM  |                                          |   0.1%
  GBM  | █                                        |   4.3%
  RF   | █                                        |   2.5%
  AVG  |                                          |   2.3%  (ensemble)

  Verdict: GOOD — consistent with intact training samples.
